## tl;dr

V11.20 frozen-policy diagnosis; not a new Alpha or a refit. Both original primary candidates remain failed.


## Context & Methods

12 zero-fee replays versus24 immutable paid paths,2023–2024 reused history,CNY3m continuous.
### Key Assumptions

Targets are byte-frozen;cash compounding means fee addback is not a zero-cost run. Original16models/32fits are inherited;newfits0. No2025/26.
RequiresPython3.10+ andDuckDB;local ignored operation files are required for full reproduction.


## Data

### 1. Verify immutable evidence and debt


In [1]:
import json,hashlib,math
from pathlib import Path
import duckdb
repo=Path.cwd()
if not (repo/'docs/V11_20_RESULT.summary.json').exists(): repo=repo.parent
operation=repo/'artifacts/gross-net/epoch-001'
summary=json.loads((repo/'docs/V11_20_RESULT.summary.json').read_text(encoding='utf-8'))
audit=json.loads((operation/'INDEPENDENT_AUDIT.json').read_text(encoding='utf-8'))
result=json.loads((operation/'RESULT.json').read_text(encoding='utf-8'))
assert audit['pass'] and summary['independent_audit_pass'] and not summary['validated_alpha']
assert hashlib.sha256((operation/'RESULT.json').read_bytes()).hexdigest()==summary['source_result_sha256']
assert len(summary['rows'])==36 and summary['raw_trial_lower_bound']==3660
print({'accounts':36,'debt':3660,'new_fits':summary['new_fits'],'inherited_fits':summary['inherited_fit_receipts']})


{'accounts': 36, 'debt': 3660, 'new_fits': 0, 'inherited_fits': 32}


### 2. Check source query and frozen target byte identity without rerunning source reads


In [2]:
assert (repo/'scripts/gross_net_source_audit.sql').read_text(encoding='utf-8')==audit['source_query']
parent=Path(result['spec']['parent_dir'])
for plan in result['spec']['plans']:
    rel='targets/'+plan['target_key']+'.json'
    assert (operation/rel).read_bytes()==(parent/rel).read_bytes()
assert summary['new_fits']==0 and summary['inherited_models']==16
print({'frozen_target_sets':12,'same_source_query':True})


{'frozen_target_sets': 12, 'same_source_query': True}


## Results

### 3. Independently recompute all36 saved account aggregates


In [3]:
actual={}
template=(repo/'scripts/lead_challenge_audit.sql').read_text(encoding='utf-8')
for folder in (operation,parent):
    with duckdb.connect() as con:
        cur=con.execute(template.replace('__ACCOUNT_GLOB__',(folder/'accounts/*.jsonl').as_posix()))
        names=[d[0] for d in cur.description]
        for r in cur.fetchall():
            assert r[0] not in actual
            actual[r[0]]=dict(zip(names,r))
assert len(actual)==36
for row in summary['rows']:
    for k in ('net_return','final_nav','cost_cny','max_drawdown'):
        assert math.isclose(row[k],actual[row['account_key']][k],rel_tol=1e-10,abs_tol=1e-6)
print({'independent_saved_account_aggregates':36,'pass':True})


{'independent_saved_account_aggregates': 36, 'pass': True}


### 4. Review primary paths and complete comparisons


In [4]:
for row in summary['rows']:
    if row['policy']=='full':
        print({k:row[k] for k in ('account_key','return2023','return2024','net_return','cost_cny')})
assert len(summary['attribution']['increments'])==42
assert len(summary['attribution']['path_drag'])==24
assert result['statistics']=={'dsr':None,'pbo':None,'placebo':None,'status':'NOT_RUN'}
assert result['screen_survived']=={'linear':False,'quadratic':False}
print({'diagnostic_only':True,'validated_alpha':False})


{'account_key': 'linear-full-0', 'return2023': 0.022496220594552474, 'return2024': 0.022670990490362364, 'net_return': 0.04567722268808638, 'cost_cny': 0.0}
{'account_key': 'linear-full-164', 'return2023': -0.15413666257017977, 'return2024': -0.1605295599307529, 'net_return': -0.2899227318893476, 'cost_cny': 927686.1517576501}
{'account_key': 'linear-full-82', 'return2023': -0.069007060390054, 'return2024': -0.07322115688339048, 'net_return': -0.137175440478563, 'cost_cny': 505487.27011722105}
{'account_key': 'quadratic-full-0', 'return2023': 0.01461518917905913, 'return2024': -0.02625148232844754, 'net_return': -0.012019963529850353, 'cost_cny': 0.0}
{'account_key': 'quadratic-full-164', 'return2023': -0.16100111915575732, 'return2024': -0.2001164894083618, 'net_return': -0.32889862980785234, 'cost_cny': 936510.0428926975}
{'account_key': 'quadratic-full-82', 'return2023': -0.07586117255480507, 'return2024': -0.11514339343742686, 'net_return': -0.18226965315412913, 'cost_cny': 510867.

## Takeaways

Zero fees cannot promote a failed paid candidate. Gross/net cross-differences are descriptive,not causal or independentOOS. Register a genuinely different follow-on mechanism or train-prefix objective; do not retune these targets.
Cells executed under ordinaryPython,not a Jupyter kernel. Jupyter frontend QA is NOT_RUN.
Optional dependency availability at build: {"nbformat": false, "nbclient": false, "ipykernel": false}
